<a href="https://colab.research.google.com/github/madhusuraj/Madhu-Suraj-Twitter/blob/main/CAIS_Break_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install keras
import pandas as pd      
trainDf = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datatwitter.csv') #csv file stored in data frame variable


 #storing the individual columns in trainDf (tweets and labels) in variables. I pass these vars as parameters to loadTweets() function, since labels and tweets are not separate files 
tweetList = trainDf.tweet
labelList = trainDf.valence

#to obtain vector rep of word inputs 
embeddings = '/content/drive/MyDrive/glove.6B.50d.txt'

In [ ]:
#function to preprocess data for RNN, taking inputs of tweets and labels
import numpy as np
from keras.preprocessing.text import text_to_word_sequence
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
import os

EMBEDDING_DIM = 50

#function adapted from CAIS++ Curriculum RNN (Week 6) load tweets example. loading tweets file to pickle was removed, as input is already a pandas dataframe rather than a file. 

def loadTweets(tweetsCol, labelsCol, embeddings_dir):

	# Tweets are tokenized 
	print("2 -- Tokenizing the tweets: converting sentences to sequence of words")
	tokenizer = Tokenizer()
	tokenizer.fit_on_texts(tweetList)

	sequences = tokenizer.texts_to_sequences(tweetList)
	word_index = tokenizer.word_index

	# Pad sequences to ensure samples are the same size
	print("3 -- Padding sequences to ensure samples are the same size")
	training_data = pad_sequences(sequences)

	print("4 -- Loading pre-trained word embeddings. This may take a few minutes.")

	embeddings_index = {}
	f = open(embeddings,'rb')
	for line in f:
	    values = line.split()
	    word = values[0].decode('UTF-8')
	    coefs = np.asarray(values[1:], dtype='float32')
	    embeddings_index[word] = coefs
	f.close()

	print("5 -- Finding word embeddings for words in our tweets.")
 
	# prepare word embedding matrix
	num_words = len(word_index)+1
	embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))
	for word, i in word_index.items():
	    if i >= num_words:
	        continue
	    embedding_vector = embeddings_index.get(word)
	    if embedding_vector is not None:
	        # words not found in embedding index will be all-zeros.
	        embedding_matrix[i] = embedding_vector

	return tweetList, training_data, labelList, word_index, embedding_matrix

In [ ]:
tweets, tweets_preprocessed, labels, word_index, embedding_matrix = loadTweets(tweetList, labelList, embeddings)

2 -- Tokenizing the tweets: converting sentences to sequence of words
3 -- Padding sequences to ensure samples are the same size
4 -- Loading pre-trained word embeddings. This may take a few minutes.
5 -- Finding word embeddings for words in our tweets.


In [ ]:
print(tweets_preprocessed)
print(labels); #these are the key inputs to rnn

[[     0      0      0 ...     41      9    385]
 [     0      0      0 ...     40    273   1170]
 [     0      0      0 ...     31     12  27341]
 ...
 [     0      0      0 ...     14     11   2107]
 [     0      0      0 ...  13870 131975  98577]
 [     0      0      0 ... 230191 690959 690960]]
0          0
1          0
2          0
3          0
4          0
          ..
1599995    4
1599996    4
1599997    4
1599998    4
1599999    4
Name: valence, Length: 1600000, dtype: int64


In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, Input
from keras.layers.merge import Concatenate
from keras.layers.core import Dense, Activation, Flatten
from keras.layers import Dropout, concatenate
from keras.layers.recurrent import LSTM
from keras.layers.wrappers import Bidirectional
from keras.layers.convolutional import Conv1D, MaxPooling1D
from keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard
from keras import metrics
from keras.models import Model

In [ ]:
model = Sequential()

#inputs words --> converted to GloVe vectors 
model.add(Embedding(len(word_index) + 1, EMBEDDING_DIM, weights=[embedding_matrix], input_length=tweets_preprocessed.shape[1], trainable=False)) 

# return sequence is set to true here, the output is passed into second LSTM layer
model.add(LSTM(64, return_sequences = True, activation='relu'))
model.add(Dropout(.2))

model.add(LSTM(64, activation='relu'))
model.add(Dropout(.2))

model.add(Dense(32, activation='relu'))
model.add(Dropout(.2))

model.add(Dense(1, activation = 'sigmoid'))



print(model.summary())

OPTIMIZER = 'RMSprop' #chosen as best suited for RNN 

LOSS = 'categorical_crossentropy' # because we're classifying with 3 categories

#valence is an ordinal categorical variable --> sparse categorical accuracy use. not one hot

model.compile(loss = LOSS, optimizer = OPTIMIZER, metrics = [metrics.sparse_categorical_accuracy])

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 118, 50)           34548050  
                                                                 
 lstm_4 (LSTM)               (None, 118, 64)           29440     
                                                                 
 dropout_6 (Dropout)         (None, 118, 64)           0         
                                                                 
 lstm_5 (LSTM)               (None, 64)                33024     
                                                                 
 dropout_7 (Dropout)         (None, 64)                0         
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dropout_8 (Dropout)         (None, 32)               

In [ ]:
model.fit(tweets_preprocessed, labels, 
          epochs = 5, 
          batch_size = 128, 
          validation_split =0.5)

Epoch 1/5
 727/6250 [==>...........................] - ETA: 34:27 - loss: nan - sparse_categorical_accuracy: 1.0000

KeyboardInterrupt: ignored